# Spine XR Augmentation — Colab Runner

Bu notebook Google Colab Pro+ (A100) üzerinde çalışacak şekilde tasarlandı. Tüm ağır eğitim buradan koşturulur.

## Sıralama

1. Bootstrap (Drive mount, repo clone, pip install, StyleGAN2-ADA clone)
2. `01_audit` → `02_splits`
3. `03_train_classifier --config baseline.yaml --fold {1,2,3}` + `03b_summarize_cv`
4. `03_train_classifier --config traditional.yaml --fold {1,2,3}` + `03b_summarize_cv`
5. StyleGAN: `04a_prepare_gan_pool` → `04b_train_stylegan --stage base` → `--stage per_class --class <name>` × 7 → `04c_generate_stylegan`
6. LDM: `06_train_ldm --config vae.yaml` → `06_train_ldm --config ldm.yaml` → `07_generate_ldm`
7. `03_train_classifier --config stylegan_aug.yaml --fold {1,2,3}` + `ldm_aug.yaml` + `combined.yaml`
8. `09_test_eval --configs ...` → `10_make_report`

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları ve Ağırlıkları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adını "final_preprocessed_data" Yerine "dataset" Yap
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr/dataset.rar') 
!unrar x {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs'):
    shutil.rmtree('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
# StyleGAN2-ADA NVIDIA repo clone
!mkdir -p third_party && cd third_party && [ -d stylegan2-ada-pytorch ] || git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

## 2. Audit + Splits

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_splits.py --config configs/base.yaml
!cat outputs/01_audit/audit_report.md

## 3. Baseline classifier — 3 fold

### Smoke Test

In [ ]:
# SMOKE test önce (3 epoch fold 1)
!python scripts/03_train_classifier.py --config configs/classifier/baseline.yaml --fold 1 --smoke

### Fold 1

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/baseline.yaml --fold 1

### Fold 2

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/baseline.yaml --fold 2

### Fold 3

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/baseline.yaml --fold 3

### Sonuçları Özetle (3 Fold Tamamlandıktan Sonra)

In [ ]:
!python scripts/03b_summarize_cv.py --config configs/classifier/baseline.yaml

## 4. Traditional

### Fold 1

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/traditional.yaml --fold 1

### Fold 2

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/traditional.yaml --fold 2

### Fold 3

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/traditional.yaml --fold 3

### Sonuçları Özetle (3 Fold Tamamlandıktan Sonra)

In [ ]:
!python scripts/03b_summarize_cv.py --config configs/classifier/traditional.yaml

## 5. StyleGAN2-ADA

### Hazırlık

In [ ]:
!python scripts/04a_prepare_gan_pool.py

### PyTorch 2.4 Uyum Patch'i (Base/Per-class eğitiminden önce ÇALIŞTIR)

NVIDIA `stylegan2-ada-pytorch` repo'su PyTorch 1.7 dönemine ait. Colab'in PyTorch 2.4.1 + Py3.12 ortamında `grid_sample_gradfix` / `conv2d_gradfix` özel autograd path'leri `TypeError: 'tuple' object is not callable` ile patlıyor; ayrıca C++ plugin build'leri için A100 sm_80 arch ve `MAX_JOBS` sınırı gerekiyor. Aşağıdaki hücre her iki gradfix dosyasını `enabled = False` olarak yamalayıp gerekli env değişkenlerini set eder. **Idempotent** — birden fazla çalıştırılabilir.

In [ ]:
# === StyleGAN2-ADA — PyTorch 2.4 uyum patch'i (Colab A100, Python 3.12) ===
# Tek seferlik. Base/per-class eğitim hücrelerinden ÖNCE çalıştır.
# Yaptığı: (1) C++ extension build env'leri, (2) gradfix dosyalarında enabled=False.
import os, pathlib, re, sys

os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0"          # A100 = sm_80
os.environ["MAX_JOBS"] = "4"                        # Colab RAM koruması (ninja)
os.environ["TORCH_EXTENSIONS_DIR"] = "/root/.cache/torch_extensions_sg2"
pathlib.Path(os.environ["TORCH_EXTENSIONS_DIR"]).mkdir(parents=True, exist_ok=True)

SG_ROOT = pathlib.Path("third_party/stylegan2-ada-pytorch")
assert SG_ROOT.exists(), f"StyleGAN repo bulunamadı: {SG_ROOT}. Bootstrap clone hücresini çalıştır."

# grid_sample_gradfix ve conv2d_gradfix → enabled=False (PyTorch 2.x'te kırık olan
# özel autograd path'leri yerine native PyTorch backward kullanılır).
for fname in ("grid_sample_gradfix.py", "conv2d_gradfix.py"):
    p = SG_ROOT / "torch_utils" / "ops" / fname
    src = p.read_text()
    patched = re.sub(
        r"^enabled\s*=\s*True\s*$",
        "enabled = False  # patched: PyTorch 2.x uyumu",
        src, count=1, flags=re.MULTILINE,
    )
    if patched == src:
        if "enabled = False" in src:
            print(f"  • {fname} zaten patched, atlanıyor.")
            continue
        raise RuntimeError(f"Patch hedefi (`enabled = True`) bulunamadı: {p}")
    p.write_text(patched)
    print(f"  ✓ patched {fname}")

# Sanity: patch tutmuş mu?
sys.path.insert(0, str(SG_ROOT))
from torch_utils.ops import grid_sample_gradfix, conv2d_gradfix
assert grid_sample_gradfix.enabled is False
assert conv2d_gradfix.enabled is False
print("StyleGAN2-ADA patch hazır. Eğitim başlatılabilir.")

### Base Eğitim

In [ ]:
!python scripts/04b_train_stylegan.py --stage base

### Sınıf 1 (Osteophytes)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Osteophytes"

### Sınıf 2 (Disc space narrowing)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Disc space narrowing"

### Sınıf 3 (Other lesions)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Other lesions"

### Sınıf 4 (Foraminal stenosis)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Foraminal stenosis"

### Sınıf 5 (Surgical implant)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Surgical implant"

### Sınıf 6 (Spondylolysthesis)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Spondylolysthesis"

### Sınıf 7 (Vertebral collapse)

In [ ]:
!python scripts/04b_train_stylegan.py --stage per_class --class "Vertebral collapse"

### Üretim

In [ ]:
!python scripts/04c_generate_stylegan.py

## 6. Latent Diffusion

### VAE Eğitimi

In [ ]:
!python scripts/06_train_ldm.py --config configs/diffusion/vae.yaml

### LDM Eğitimi

In [ ]:
!python scripts/06_train_ldm.py --config configs/diffusion/ldm.yaml

### Sentetik Resim Üretimi

In [ ]:
!python scripts/07_generate_ldm.py

## 7. Augmented classifier runs

### Metadata Üretme

In [ ]:
!python scripts/08_build_combined_metadata.py

### StyleGAN Eğitimleri

#### Fold 1

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/stylegan_aug.yaml --fold 1

#### Fold 2

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/stylegan_aug.yaml --fold 2

#### Fold 3

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/stylegan_aug.yaml --fold 3

#### Sonuçları Özetle (3 Fold Tamamlandıktan Sonra)

In [ ]:
!python scripts/03b_summarize_cv.py --config configs/classifier/stylegan_aug.yaml

### LDM Eğitimleri

#### Fold 1

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/ldm_aug.yaml --fold 1

#### Fold 2

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/ldm_aug.yaml --fold 2

#### Fold 3

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/ldm_aug.yaml --fold 3

#### Sonuçları Özetle (3 Fold Tamamlandıktan Sonra)

In [ ]:
!python scripts/03b_summarize_cv.py --config configs/classifier/ldm_aug.yaml

### Combined Eğitimleri

#### Fold 1

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/combined.yaml --fold 1

#### Fold 2

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/combined.yaml --fold 2

#### Fold 3

In [ ]:
!python scripts/03_train_classifier.py --config configs/classifier/combined.yaml --fold 3

#### Sonuçları Özetle (3 Fold Tamamlandıktan Sonra)

In [ ]:
!python scripts/03b_summarize_cv.py --config configs/classifier/combined.yaml

## 8. Test evaluation + Final report

In [ ]:
!python scripts/09_test_eval.py --configs configs/classifier/baseline.yaml configs/classifier/traditional.yaml configs/classifier/stylegan_aug.yaml configs/classifier/ldm_aug.yaml configs/classifier/combined.yaml
!python scripts/10_make_report.py --config configs/base.yaml
!cat outputs/final_report.md | head -200